# Praktikum 8: Reaalajas andmetöötlus. Spark Structured Streaming + Kafka

Notebook on jagatud nelja progresseeruvasse ossa:

1. **Kafka alused.** Brokeri käivitamine, teema (_topic_) loomine, sõnumite tootmine ja tarbimine, partitsiooni järjekorra ja tarbijagrupi mahajäämuse vaatlemine.
2. **Struktureeritud voogtöötlus algusest lõpuni.** `readStream.format("kafka")`, JSON parsimine, kirjutamine mälusihtkohta, jooksvate sõnumite tootmine.
3. **Akendega agregeerimine sündmuse aja alusel.** Kiikuvad aknad (_tumbling windows_), hilinenud andmete käitumine, vesimärgid (_watermarks_), väljundirežiimid.
4. **Keerukamad mustrid.** Kirjutamine Delta tabelisse, kontrollpunkti taaskäivitus, voo ja staatilise andmestiku liitmine.

Käivita lahtrid **järjekorras**. Iga osa toetub eelmisele.

---
## Seadistus

In [ ]:
# Paigalda Kafka klient (Pythoni teek) ja Delta Lake'i Pythoni pakett.
# delta-spark pole antud juhul vajalik, aga lihtsustab hiljem Delta tabelitega töötamist lisaülesandes.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kafka-python-ng", "delta-spark"])
print("kafka-python-ng ja delta-spark on paigaldatud")

In [ ]:
# Loome SparkSession-i koos Kafka konnektori ja Delta Lake'i toega.
# spark.jars.packages laeb vajalikud JAR-id Maven Central-ist Sparki käivitumisel.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("praktikum8_voogandmed")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

In [ ]:
# Kafka maakleri ühendusstring Dockeri sisevõrgus.
# Hostinime 'kafka' lahendab Docker DNS maakleri konteineri aadressiks.
BOOTSTRAP = "kafka:9092"
TOPIC = "sensor-events"

---
## Osa 1: Kafka alused

### 1.1. Kontrolli, et keskkond töötab

Hosti terminalis (mitte Jupyteri sees) käivita:

```bash
docker compose ps
```

Mõlemad teenused (`praktikum8_kafka` ja `praktikum8_jupyter`) peavad olema staatuses `Up`.

### 1.2. CLI viide

Allpool olevad lahtrid kasutavad Pythoni teeki `kafka-python-ng`. Sama tegevust saab teha ka käsurealt CLI-tarkvaraga (vt README jaotis "Kafka CLI viide"). CLI käskude käivitamine **ei ole** notebooki läbimiseks vajalik. Kui soovid, käivita need hosti terminalis ja vaatle, kuidas Pythoni API ja CLI samasuguse tulemuse annavad.

In [ ]:
# Loome teema kolme partitsiooniga. Sama tulemus on ka CLI käsuga
# 'kafka-topics.sh --create --topic sensor-events --partitions 3 --replication-factor 1'.
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP, client_id="praktikum8-admin")

try:
    admin.create_topics([
        NewTopic(name=TOPIC, num_partitions=3, replication_factor=1)
    ])
    print(f"Teema '{TOPIC}' loodud kolme partitsiooniga")
except TopicAlreadyExistsError:
    print(f"Teema '{TOPIC}' on juba olemas")

print("Olemasolevad teemad:", admin.list_topics())
admin.close()

In [ ]:
# Toodame üheksa võtmega sõnumit. Kolm sensorit, kolm sõnumit kummalegi.
# Kafka vaikejaotur arvutab partitsiooni võtme räsist (_hash_), seega
# sama võtmega sõnumid satuvad alati ühte ja samasse partitsiooni.
import json, random
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    key_serializer=lambda k: k.encode(),
    value_serializer=lambda v: json.dumps(v).encode(),
)

sensors = ["sensor-1", "sensor-2", "sensor-3"]
print(f"{'Partitsioon':>12} {'Nihe':>8} {'Võti':>12}  Väärtus")
print("-" * 60)
for i in range(9):
    sensor = sensors[i % 3]
    msg = {"sensor_id": sensor, "temperature": round(20 + random.uniform(-2, 2), 2)}
    future = producer.send(TOPIC, key=sensor, value=msg)
    meta = future.get(timeout=10)
    print(f"{meta.partition:>12} {meta.offset:>8} {sensor:>12}  {msg}")

producer.close()
print("Pane tähele: iga sensori sõnumid satuvad alati ühte ja samasse partitsiooni.")

In [ ]:
# Tarbime kõik sõnumid ja kuvame nende partitsiooni ja nihke.
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    auto_offset_reset="earliest",
    group_id="praktikum8-inspect-group",
    consumer_timeout_ms=4000,  # peatu, kui 4 sekundit pole uut sõnumit
    value_deserializer=lambda v: json.loads(v.decode()),
    key_deserializer=lambda k: k.decode() if k else None,
)

print(f"{'Partitsioon':>12} {'Nihe':>8} {'Võti':>12}  Väärtus")
print("-" * 60)
for msg in consumer:
    print(f"{msg.partition:>12} {msg.offset:>8} {str(msg.key):>12}  {msg.value}")

consumer.close()

In [ ]:
# Vaatleme tarbijagrupi nihkeid. Sama tulemuse annab ka CLI käsk:
#   kafka-consumer-groups.sh --describe --group praktikum8-inspect-group
admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
offsets = admin.list_consumer_group_offsets("praktikum8-inspect-group")

print(f"{'Teema/partitsioon':40} {'Salvestatud nihe':>20}")
print("-" * 62)
for tp, meta in sorted(offsets.items()):
    print(f"{str(tp):40} {meta.offset:>20}")

admin.close()
print("\nMahajäämus = viimane partitsiooni nihe - salvestatud nihe.")
print("Mahajäämus = 0 tähendab, et tarbija on järje peal.")

---
## Osa 2: Struktureeritud voogtöötlus algusest lõpuni

Eesmärk: kirjutada töötav `readStream` ja `writeStream` päring.

Kafkast loetud DataFrame'il on alati sama toorskeem (_raw schema_), olenemata teema sisust:

| Veerg | Tüüp | Selgitus |
|-------|------|----------|
| `key` | binary | sõnumi võti |
| `value` | binary | sõnumi keha. Seda parsime |
| `topic` | string | teema nimi |
| `partition` | int | partitsiooni number |
| `offset` | long | nihe partitsiooni sees |
| `timestamp` | timestamp | maakleri ajatempel |
| `timestampType` | int | 0 = CreateTime, 1 = LogAppendTime |

In [ ]:
# Loeme Kafkast voogu. DataFrame on laisk: andmed ei liigu enne, kui päring käivitub.
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")  # kaasa sõnumid, mis juba teemas on
    .load()
)

print("isStreaming:", raw_df.isStreaming)  # True. See on voo DataFrame, mitte tavaline
print()
raw_df.printSchema()

In [ ]:
# Parsime 'value' veeru: binaar -> JSON -> eraldi väljad.
# Skeemi tuleb voogtöötluses alati käsitsi anda. Skeemi tuletamist (_inference_) voo puhul ei toetata.
# Osa 1 sõnumites pole 'event_time' välja, seega see veerg on neil null.
event_schema = "sensor_id STRING, temperature DOUBLE, event_time STRING"

parsed_df = (
    raw_df.select(
        F.col("key").cast("string").alias("key"),
        F.from_json(F.col("value").cast("string"), event_schema).alias("d"),
        "partition",
        "offset",
        F.col("timestamp").alias("kafka_time"),
    )
    .select("key", "d.*", "partition", "offset", "kafka_time")
)

print("Parsitud skeem:")
parsed_df.printSchema()

In [ ]:
# Kirjutame voo mälusihtkohta. Saame seda pärida tavalise SQL-iga (spark.sql).
# Mälusihtkoht on mõeldud AINULT silumiseks. Tootmises kasuta failipõhist või Delta sihtkohti.
import time

for q in spark.streams.active:
    if q.name == "raw_events":
        q.stop()

raw_query = (
    parsed_df.writeStream
    .format("memory")
    .queryName("raw_events")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-raw")
    .trigger(processingTime="3 seconds")
    .start()
)

time.sleep(8)  # ootame kaks käivitustsüklit
print("Seni vastu võetud sõnumid:")
spark.sql("""
    SELECT key, sensor_id, temperature, event_time, partition, offset, kafka_time
    FROM raw_events
    ORDER BY partition, offset
""").show(truncate=False)

In [ ]:
# Toodame paar uut sõnumit, samal ajal kui voo päring on aktiivne.
# Pärast järgmise käivitustsükli käivitamist ilmuvad need tabelisse.
from datetime import datetime, timezone

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    key_serializer=lambda k: k.encode(),
    value_serializer=lambda v: json.dumps(v).encode(),
)
for sensor in ["sensor-1", "sensor-2", "sensor-3"]:
    msg = {
        "sensor_id": sensor,
        "temperature": round(21 + random.uniform(0, 5), 2),
        "event_time": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S"),
    }
    producer.send(TOPIC, key=sensor, value=msg)
producer.flush()
producer.close()

time.sleep(8)
print("Pärast uute sõnumite tootmist:")
spark.sql("""
    SELECT key, sensor_id, temperature, event_time, partition, offset
    FROM raw_events
    ORDER BY partition, offset
""").show(truncate=False)

In [ ]:
raw_query.stop()
print("Osa 2 päring peatatud.")

---
## Osa 3: Akendega agregeerimine sündmuse aja alusel

### Põhimõisted

| Mõiste | Tähendus |
|--------|----------|
| **Sündmuse aeg** (_event time_) | Ajatempel, mille tootja sõnumi **kehasse** paneb |
| **Töötluse aeg** (_processing time_) | Süsteemiaeg, mil Spark sõnumi vastu võtab |
| **Kattuvuseta aken** (_tumbling window_) | Sama suurusega kattumata aknad. Iga sündmus kuulub täpselt ühte aknasse |
| **Vesimärk** (_watermark_) | Määrab, kui kaua Spark hilinenud sündmusi ootab. Vanemad aknad visatakse välja |
| **Väljundirežiim** (_output mode_) | `append`: ainult uued read. `complete`: kogu tulemustabel iga trigeriga. `update`: ainult muutunud read |

Ilma vesimärgita säilitab Spark iga akna olekut igavesti (õige tulemus, kuid mälukulu kasvab).
Vesimärgi *T* korral viskab Spark välja akna, mille lõpp on `max(sündmuse_aeg) - T`-st vanem.

In [ ]:
# Loeme voo uuesti, ainult uutest sõnumitest ('latest').
# Teisendame event_time veeru TIMESTAMP tüübiks, et akenduse funktsioonid töötaks.
sensor_schema = "sensor_id STRING, temperature DOUBLE, event_time STRING"

stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
    .select(
        F.from_json(F.col("value").cast("string"), sensor_schema).alias("d")
    )
    .select(
        "d.sensor_id",
        "d.temperature",
        F.to_timestamp("d.event_time", "yyyy-MM-dd'T'HH:mm:ss").alias("event_time"),
    )
    .filter(F.col("event_time").isNotNull())  # jätame välja Osa 1 sõnumid
)

stream_df.printSchema()

In [ ]:
# Abifunktsioon: toodab sündmusi juhitava sündmuse aja nihkega.
# time_offset_minutes < 0  -> sündmuse aeg minevikus (hilinenud sündmused)
# time_offset_minutes = 0  -> sündmuse aeg on 'praegu'
from datetime import datetime, timezone, timedelta

def produce_events(n=5, time_offset_minutes=0, temperature_base=22.0):
    p = KafkaProducer(
        bootstrap_servers=BOOTSTRAP,
        key_serializer=lambda k: k.encode(),
        value_serializer=lambda v: json.dumps(v).encode(),
    )
    ts = (datetime.now(timezone.utc) + timedelta(minutes=time_offset_minutes)).strftime(
        "%Y-%m-%dT%H:%M:%S"
    )
    for i in range(n):
        sensor = f"sensor-{(i % 3) + 1}"
        msg = {
            "sensor_id": sensor,
            "temperature": round(temperature_base + random.uniform(-1, 1), 2),
            "event_time": ts,
        }
        p.send(TOPIC, key=sensor, value=msg)
    p.flush()
    p.close()
    print(f"Toodetud {n} sündmust, sündmuse aja nihe = {time_offset_minutes:+d} min ({ts})")

### 3.1. Kattuvuseta aken `complete` režiimis ilma vesimärgita

Ilma vesimärgita säilitab Spark iga akna olekut igavesti.
Hilinenud sündmus tekitab vana akna ümberarvutuse ja see ilmub uuesti tulemustesse.
Allpool olevad lahtrid näitavad seda käitumist.

In [ ]:
for q in spark.streams.active:
    if q.name == "windowed_complete":
        q.stop()

windowed_df = (
    stream_df
    .groupBy(
        F.window("event_time", "1 minute").alias("window"),
        "sensor_id",
    )
    .agg(
        F.round(F.avg("temperature"), 2).alias("avg_temp"),
        F.count("*").alias("n"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "sensor_id", "avg_temp", "n",
    )
)

windowed_query = (
    windowed_df.writeStream
    .format("memory")
    .queryName("windowed_complete")
    .outputMode("complete")
    .option("checkpointLocation", "/tmp/chk-windowed-complete")
    .trigger(processingTime="5 seconds")
    .start()
)

print("Akenduse päring käivitatud (complete režiim, vesimärki ei ole).")

In [ ]:
# Toodame normaalse aja sündmusi (event_time = praegu) ja vaatame akna olekut.
produce_events(n=6, time_offset_minutes=0, temperature_base=22.0)
time.sleep(10)

spark.sql("""
    SELECT window_start, window_end, sensor_id, avg_temp, n
    FROM windowed_complete
    ORDER BY window_start, sensor_id
""").show(truncate=False)

In [ ]:
# Toodame hilinenud sündmused, 10 minutit minevikus.
# Ilma vesimärgita ei ole Sparkil mingit alust vanu aknaid kustutada.
# 10 minuti tagune aken tekitatakse ja kuvatakse complete tulemuses.
# (temperature_base=99.9 teeb need read kergesti silmaga eristatavaks)
produce_events(n=3, time_offset_minutes=-10, temperature_base=99.9)
time.sleep(10)

print("Pärast hilinenud sündmusi (vana aken ilmub, sest vesimärki ei ole):")
spark.sql("""
    SELECT window_start, window_end, sensor_id, avg_temp, n
    FROM windowed_complete
    ORDER BY window_start, sensor_id
""").show(truncate=False)

In [ ]:
windowed_query.stop()
print("complete režiimi päring peatatud.")

### 3.2. Kattuvuseta aken `update` režiimis koos vesimärgiga

**Vesimärk** 5 minutit ütleb Sparkile:
> "Kui maksimaalne nähtud sündmuse aeg ületab akna lõppu rohkem kui 5 minutiga,
> ei tule sellesse aknasse enam andmeid. Vabasta selle akna olek."

Hilinenud sündmused, mis langevad vesimärgi piirist välja, visatakse vaikimisi kõrvale.

**`update` väljundirežiim** kirjutab välja ainult need read, mis igal trigeril muutusid.
Akenduse agregeerimisel koos vesimärgiga on `update` enamasti efektiivsem kui `complete`.

In [ ]:
for q in spark.streams.active:
    if q.name == "windowed_update":
        q.stop()

watermarked_df = (
    stream_df
    .withWatermark("event_time", "5 minutes")
    .groupBy(
        F.window("event_time", "1 minute").alias("window"),
        "sensor_id",
    )
    .agg(
        F.round(F.avg("temperature"), 2).alias("avg_temp"),
        F.count("*").alias("n"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "sensor_id", "avg_temp", "n",
    )
)

watermarked_query = (
    watermarked_df.writeStream
    .format("memory")
    .queryName("windowed_update")
    .outputMode("update")
    .option("checkpointLocation", "/tmp/chk-windowed-update")
    .trigger(processingTime="5 seconds")
    .start()
)

print("Akenduse päring käivitatud (update režiim, vesimärk = 5 min).")

In [ ]:
# Toodame normaalseid sündmusi ja vaatleme jooksvat akent.
produce_events(n=6, time_offset_minutes=0, temperature_base=22.0)
time.sleep(10)

print("Praegused aknad (update režiim, vesimärk = 5 min):")
spark.sql("""
    SELECT window_start, window_end, sensor_id, avg_temp, n
    FROM windowed_update
    ORDER BY window_start, sensor_id
""").show(truncate=False)

In [ ]:
# Toodame väga hilinenud sündmused, 15 minutit minevikus.
# Vesimärk = 5 min, seega 15 min tagused sündmused on piirist väljas.
# Spark viskab need vaikselt kõrvale. Vana akna read EI ilmu tulemustes.
produce_events(n=3, time_offset_minutes=-4, temperature_base=99.9)
time.sleep(10)

print("Pärast väga hilinenud sündmusi (vesimärk = 5 min, hilinemine = 15 min):")
spark.sql("""
    SELECT window_start, window_end, sensor_id, avg_temp, n
    FROM windowed_update
    ORDER BY window_start, sensor_id
""").show(truncate=False)

print("\nTähele: avg_temp ~ 99.9 ridu pole. Hilinenud sündmused visati kõrvale.")

In [ ]:
# Peatame kõik jooksvad päringud enne Osa 4 algust.
for q in spark.streams.active:
    q.stop()
print("Kõik päringud peatatud.")

---
## Osa 4: Keerukamad mustrid

Allpool olevad mustrid on aluseks medaljonarhitektuuriga voo puhul.

### 4.1. Kirjutamine Delta tabelisse ja kontrollpunkti taaskäivituse korrektsus

Toodangus kirjutate enamasti tabelisse (Delta, Iceberg, ...), mitte memory-sse.
Kriitiline garantii on:

> Pärast taaskäivitust loeb päring **kontrollpunkti salvestatud nihetelt**, mitte
> `startingOffsets`-ist. Sõnumid, mis on juba kirjutatud, ei satu uuesti väljundisse.

Allpool uurid seda enne ja pärast taaskäivitust ridade arvu võrdluse abil.

In [ ]:
import shutil

# Puhastame eelmise käivituse jälgi
shutil.rmtree("/tmp/bronze", ignore_errors=True)
shutil.rmtree("/tmp/chk-bronze", ignore_errors=True)

# Sama loogika kui Osas 2, ainult 'startingOffsets' = 'earliest' (alustame algusest).
bronze_source = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(
        F.col("value").cast("string").alias("raw_value"),
        "partition",
        "offset",
        F.col("timestamp").alias("kafka_time"),
    )
)

# Kirjutame toorsündmused Delta tabelisse. See on pronkskihi (_bronze layer_) muster.
# checkpointLocation hoiab arvet, millised nihked on juba töödeldud.
bronze_query = (
    bronze_source.writeStream
    .format("delta")
    .outputMode("append")
    .option("path", "/tmp/bronze")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .start()
)

# Toodame paar uut sündmust jooksvale päringule otsa.
produce_events(n=4, time_offset_minutes=0)

time.sleep(15)  # ootame vähemalt kaks trigerit

count_before = spark.read.format("delta").load("/tmp/bronze").count()
print(f"Ridade arv enne taaskäivitust: {count_before}")

bronze_query.stop()
print("Päring peatatud.")

In [ ]:
# Taaskäivitame päringu SAMA kontrollpunktiga.
# Kuigi 'startingOffsets' on endiselt 'earliest', ignoreerib Spark seda valikut
# ja jaotab oma töö kontrollpunktist edasi.
bronze_query2 = (
    bronze_source.writeStream
    .format("delta")
    .outputMode("append")
    .option("path", "/tmp/bronze")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(12)

count_after = spark.read.format("delta").load("/tmp/bronze").count()
print(f"Ridade arv enne  : {count_before}")
print(f"Ridade arv pärast: {count_after}")
if count_after == count_before:
    print("\nArvud on võrdsed. Kontrollpunkt välistas juba töödeldud sõnumite korduva töötlemise.")
else:
    print(f"\nVahe = {count_after - count_before} (need on uued sõnumid kahe käivituse vahel).")

bronze_query2.stop()

In [ ]:
# Delta tabel hoiab versiooniajaloo. DESCRIBE HISTORY näitab kõiki muudatusi.
spark.sql("DESCRIBE HISTORY delta.`/tmp/bronze`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

# Ajas tagasiminek: loeme tabeli versiooni 0 (esimese commit-i järgne seis).
v0_count = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load("/tmp/bronze")
    .count()
)
print(f"\nRidade arv versioonis 0: {v0_count}")
print(f"Ridade arv praeguses versioonis: {count_after}")

### 4.2. Voo ja staatilise andmestiku liitmine (rikastamine)

**Voo ja staatilise andmestiku liitmine** ühendab voo DataFrame'i ühekordselt loetud
staatilise tabeliga. Staatiline pool jagatakse (_broadcast_) kõikidele töötajatele
ja hoitakse mälus päringu eluea jooksul.

Selle mustri abil rikastame sündmusi otsingutabeliga (näiteks sensor -> asukoht).

Olulised omadused:
- Staatiline DataFrame loetakse **üks kord** päringu käivitumisel. Lähteandmete
  hilisemad muudatused jooksvasse päringusse ei jõua.
- Liitmise tüüp (`left`, `inner`, ...) määrab, mis juhtub voo ridadega, millele
  staatilises tabelis vastet ei leita.
- Vesimärki ei vaja, sest staatilisel poolel pole sündmuse aega.

In [ ]:
# Staatiline otsingutabel. Tegelikult loetaks see CSV- või Parquet-failist.
sensor_lookup = spark.createDataFrame(
    [
        ("sensor-1", "Hoone A", "sees"),
        ("sensor-2", "Hoone B", "sees"),
        ("sensor-3", "Katus",   "väljas"),
    ],
    schema="sensor_id STRING, location STRING, sensor_type STRING",
)
sensor_lookup.show()

In [ ]:
for q in spark.streams.active:
    if q.name == "enriched_events":
        q.stop()

# Liidame voo (vasak pool) staatilise otsingutabeliga (parem pool).
# 'left' liitmine: kui staatilises tabelis vastet pole, jäävad veerud null-iks.
enriched_df = (
    stream_df.join(F.broadcast(sensor_lookup), on="sensor_id", how="left")
)

enriched_query = (
    enriched_df.writeStream
    .format("memory")
    .queryName("enriched_events")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-enriched")
    .trigger(processingTime="5 seconds")
    .start()
)

produce_events(n=6, time_offset_minutes=0)
time.sleep(12)

print("Rikastatud voog: sensor_id liidetud asukoha ja tüübiga:")
spark.sql("""
    SELECT sensor_id, temperature, event_time, location, sensor_type
    FROM enriched_events
    ORDER BY event_time, sensor_id
""").show(truncate=False)

enriched_query.stop()

In [ ]:
# Peatame kõik jooksvad päringud enne ülesannete osa.
for q in spark.streams.active:
    q.stop()
print("Kõik päringud peatatud.")

---
## Ülesanded

Kasuta vabalt dokumentatsiooni, näited demo-osast ja AI-tööriistu.
Iga ülesande näidislahendus on hiljem GitHubist leitav.

---
### Ülesanne 1: Libisev aken vesimärgiga

Kirjuta voo päring, mis:

1. Loeb teemast `sensor-events` (`startingOffsets="latest"`).
2. Rakendab **libisevat akent** suurusega **2 minutit** ja sammuga **30 sekundit** veerule `event_time`.
3. Grupeerib `sensor_id` järgi ja arvutab `min(temperature)`, `max(temperature)`, `count`.
4. Lisab vesimärgi **3 minutit**.
5. Kirjutab mälusihtkohta `update` režiimis.
6. Toodab 10 sündmust ja kuvab tulemustabeli.
7. Toodab 5 hilinenud sündmust (5 minutit minevikus, vesimärgi piires) ja kuvab tabeli uuesti.
8. Peatab voo.

**Vihje.** `F.window(time_col, windowDuration, slideDuration)` võtab kolmandaks argumendiks libiseva sammu.

**Arutelu.** Mitu aknarida tekib ühe sündmuse kohta? Miks? Millal eelistada libisevat akent kattuvuseta aknale?

---
### Ülesanne 2: Väljundirežiimi võrdlus

Käivita sama 1-minutiline kattuvuseta akenduse päring kahel korral, mõlemal vesimärk 5 minutit:

- üks `complete` väljundirežiimis (näiteks päringu nimi `windowed_complete_ex`),
- teine `update` väljundirežiimis (`windowed_update_ex`).

Sammud:
1. Tooda 6 sündmust käesolevas minuti aknas.
2. Kuva mõlemast memory tabelist tulemus.
3. Tooda 3 sündmust juurde **samasse minuti aknasse**.
4. Kuva mõlemast tabelist tulemus uuesti.

**Arutelu (vasta markdown-lahtris).**
- Mitu rida on igas tabelis pärast 4. sammu?
- Kumb režiim väljastab muutumatuid aknaid uuesti?
- Millal eelistada `update` režiimi tootmises?

---
### Ülesanne 3: Delta sihtkoht ja agregaadi salvestamine

Ehita kahekihiline andmetoru:

1. **Hõbedakiht** (_silver_): liida voog staatilise otsingutabeliga (nagu demo osas 4.2). Kirjuta rikastatud sündmused Delta tabelisse `/tmp/silver_events` koos kontrollpunktiga `/tmp/chk-silver`.
2. **Kuldkiht** (_gold_): kasuta `foreachBatch` mustrit. Iga mikropartii (_micro-batch_) sees:
   - arvuta sensori ja minuti kaupa kokkuvõte (`avg(temperature)`, `count`),
   - tee `MERGE` Delta tabelisse `/tmp/gold_aggregates` (uuendus olemasoleva (sensor, minut) puhul, lisamine uue puhul).
3. Peata mõlemad päringud. Käivita silverkihi päring sama kontrollpunktiga uuesti ja kontrolli, et `silver_events` ridade arv ei muutunud.
4. Loe `gold_aggregates` ja kuva tulemused. Käivita ka `spark.sql("DESCRIBE HISTORY delta.\`/tmp/gold_aggregates\`")` ja vaata versioonide ajalugu.

**Vihje.** Delta MERGE Pythoni API kasutamiseks impordi `from delta.tables import DeltaTable`. Esimesel käivitusel pead `gold_aggregates` tabeli kas käsitsi tühjana looma või kontrollima `DeltaTable.isDeltaTable(spark, path)` väärtust.

**Arutelu.** Miks `gold` tabelis kasutame `MERGE`-i, mitte `append`-i? Mis juhtuks, kui `silver_events` kontrollpunkt kustutada enne taaskäivitust?

---
## Edasilugemiseks

- [Structured Streaming + Kafka Integration Guide](https://spark.apache.org/docs/latest/structured-streaming-kafka-integration.html)
- [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html). Eriti sündmuse aja, vesimärkide ja väljundirežiimide jaotised.
- [PySpark Streaming API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/index.html)
- [Apache Kafka dokumentatsioon](https://kafka.apache.org/documentation/)
- [Delta Lake](https://delta.io/) ja [ACID-tehinguid voogtöötluses](https://docs.delta.io/latest/delta-streaming.html)